# Forensic Binary File Identification: Magic Bytes, MIME Types & Container Offsets
**Low-Level Signature Matching, ISOBMFF Box Navigation & Anti-Spoofing Protocols**

*Published by the [KaijuConverter Engineering Research Group](https://kaijuconverter.com)*

Relying on file extensions (`.png`, `.pdf`, `.mp4`) for content verification poses critical security hazards in file ingestion pipelines (e.g., MIME-confusion attacks, polyglot payloads, and remote code execution). Deterministic identification requires examining fixed magic byte sequences and structural container headers.

### Relevant Technical Hubs & Online Converters
- [Complete File Formats Encyclopedia](https://kaijuconverter.com/formats)
- [HEIC High-Efficiency Format Architecture](https://kaijuconverter.com/formats/heic)
- [AVIF Modern Media Container Details](https://kaijuconverter.com/formats/avif)
- [WebP Specification & Header Analysis](https://kaijuconverter.com/formats/webp)
- [FLAC Lossless Audio Stream Structure](https://kaijuconverter.com/formats/flac)
- [Apache Parquet Binary Columnar Format](https://kaijuconverter.com/formats/parquet)
- [KaijuConverter REST & MCP API Documentation](https://kaijuconverter.com/api-docs)


## 1. Magic Byte Signature Table & Offset Rules

| Format | Extension | Canonical MIME Type | Byte Sequence (Hexadecimal) | Offset | Reference Specification |
|:---|:---|:---|:---|:---:|:---|
| **PDF** | `.pdf` | `application/pdf` | `25 50 44 46 2D` (`%PDF-`) | 0 | ISO 32000-2 |
| **PNG** | `.png` | `image/png` | `89 50 4E 47 0D 0A 1A 0A` | 0 | ISO/IEC 15948 |
| **JPEG** | `.jpg` | `image/jpeg` | `FF D8 FF` | 0 | ISO/IEC 10918-1 |
| **WebP** | `.webp` | `image/webp` | `52 49 46 46 .... 57 45 42 50` (`RIFF....WEBP`) | 0, 8 | Google WebP Container |
| **AVIF** | `.avif` | `image/avif` | `.... 66 74 79 70 61 76 69 66` (`....ftypavif`) | 4, 8 | AOMedia AVIF 1.0.0 |
| **HEIC** | `.heic` | `image/heic` | `.... 66 74 79 70 68 65 69 63` (`....ftypheic`) | 4, 8 | ISO/IEC 23008-12 |
| **FLAC** | `.flac` | `audio/flac` | `66 4C 61 43` (`fLaC`) | 0 | IETF RFC 9619 |
| **Parquet** | `.parquet` | `application/vnd.apache.parquet` | `50 41 52 31` (`PAR1`) | 0 & EOF-4 | Apache Thrift Parquet |
| **SQLite** | `.sqlite` | `application/vnd.sqlite3` | `53 51 4C 69 74 65 20 66 6F 72 6D 61 74 20 33 00` | 0 | SQLite Database Format 3 |
| **ZIP** | `.zip` | `application/zip` | `50 4B 03 04` | 0 | PKWARE AppNote |
| **WASM** | `.wasm` | `application/wasm` | `00 61 73 6D` (` asm`) | 0 | W3C WebAssembly 1.0 |


In [1]:
# Production forensic byte signature detector implementation
def identify_format(header: bytes) -> dict:
    if len(header) < 16:
        return {"detected": False, "reason": "Insufficient byte stream"}

    # Fixed-offset checks
    if header.startswith(b"%PDF-"):
        return {"format": "PDF", "mime": "application/pdf", "category": "document", "doc": "https://kaijuconverter.com/formats/pdf"}
    if header.startswith(b"\x89PNG\r\n\x1a\n"):
        return {"format": "PNG", "mime": "image/png", "category": "image", "doc": "https://kaijuconverter.com/formats/png"}
    if header.startswith(b"\xff\xd8\xff"):
        return {"format": "JPEG", "mime": "image/jpeg", "category": "image", "doc": "https://kaijuconverter.com/formats/jpg"}
    if header.startswith(b"fLaC"):
        return {"format": "FLAC", "mime": "audio/flac", "category": "audio", "doc": "https://kaijuconverter.com/formats/flac"}
    if header.startswith(b"PAR1"):
        return {"format": "Parquet", "mime": "application/vnd.apache.parquet", "category": "data", "doc": "https://kaijuconverter.com/formats/parquet"}
    if header.startswith(b"SQLite format 3\x00"):
        return {"format": "SQLite", "mime": "application/vnd.sqlite3", "category": "database", "doc": "https://kaijuconverter.com/formats/sqlite"}

    # RIFF container checks
    if header.startswith(b"RIFF") and len(header) >= 12:
        riff_type = header[8:12]
        if riff_type == b"WEBP":
            return {"format": "WebP", "mime": "image/webp", "category": "image", "doc": "https://kaijuconverter.com/formats/webp"}
        if riff_type == b"WAVE":
            return {"format": "WAV", "mime": "audio/wav", "category": "audio", "doc": "https://kaijuconverter.com/formats/wav"}
        if riff_type == b"AVI ":
            return {"format": "AVI", "mime": "video/x-msvideo", "category": "video", "doc": "https://kaijuconverter.com/formats/avi"}

    # ISO Base Media File Format (ISOBMFF) ftyp box
    if len(header) >= 12 and header[4:8] == b"ftyp":
        major_brand = header[8:12].decode("ascii", errors="ignore")
        if major_brand in ["avif", "avis"]:
            return {"format": "AVIF", "mime": "image/avif", "category": "image", "doc": "https://kaijuconverter.com/formats/avif"}
        if major_brand in ["heic", "heix", "mif1", "msf1"]:
            return {"format": "HEIC", "mime": "image/heic", "category": "image", "doc": "https://kaijuconverter.com/formats/heic"}
        if major_brand in ["isom", "mp41", "mp42"]:
            return {"format": "MP4", "mime": "video/mp4", "category": "video", "doc": "https://kaijuconverter.com/formats/mp4"}

    return {"detected": False, "reason": "Unknown signature"}

# Test against synthetic headers
samples = [
    (b"%PDF-1.7\n%\xe2\xe3\xcf\xd3", "Document Upload"),
    (b"RIFF\x88\x10\x00\x00WEBPVP8 ", "Web Asset"),
    (b"\x00\x00\x00\x1cftypavif\x00\x00\x00\x00mif1", "NextGen Photo"),
    (b"PAR1\x15\x00\x15\x30", "Analytics Warehouse"),
]

for stream, label in samples:
    res = identify_format(stream)
    print(f"[{label:20s}] -> Format: {res.get('format')} | MIME: {res.get('mime')} | Link: {res.get('doc')}")


[Document Upload     ] -> Format: PDF | MIME: application/pdf | Link: https://kaijuconverter.com/formats/pdf
[Web Asset           ] -> Format: WebP | MIME: image/webp | Link: https://kaijuconverter.com/formats/webp
[NextGen Photo       ] -> Format: AVIF | MIME: image/avif | Link: https://kaijuconverter.com/formats/avif
[Analytics Warehouse ] -> Format: Parquet | MIME: application/vnd.apache.parquet | Link: https://kaijuconverter.com/formats/parquet


## 2. Integration with KaijuConverter Go & MCP SDKs

For high-throughput Go microservices, utilize our zero-dependency Go module:
```bash
go get -u github.com/PabloCirre/kaijuconverter-go
```
Or query online format specs and live transformation services via the [KaijuConverter API](https://kaijuconverter.com/api-docs).
